# Phase 2 & 3: Farm Geometry Parsing, Validation, and Spatial Visualisation
## AI-Based Multi-Temporal Remote Sensing for Crop Monitoring & Irrigation Decision Support
**Dataset:** Telangana Crop Health Challenge (GIZ-ADEX)  
**CRS:** EPSG:4326 (WGS84 Geographic) -> EPSG:32644 (UTM Zone 44N Metric)

### 1. Environment Setup & Imports
We load standard scientific Python, geospatial libraries (`geopandas`, `shapely`), and our modular project utilities.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely import make_valid, wkt

from src.data.load_data import load_train, load_test
from src.geospatial.geometry_utils import (
    add_geometry_features,
    check_geometry_validity,
    compute_area_ha,
    compute_centroids,
    compute_compactness,
    compute_perimeter_m,
    geometry_stats,
    repair_invalid_geometries,
    validate_within_telangana,
    CRS_GEO,
    CRS_METRIC,
)
from src.geospatial.spatial_viz import create_interactive_farm_map, generate_static_spatial_figure

sns.set_theme(style="darkgrid", palette="muted")
print("Libraries loaded successfully!")

### 2. Loading Train and Test Datasets
Load raw data into GeoDataFrames with CRS set to EPSG:4326.

In [ ]:
train_gdf = load_train()
test_gdf = load_test()

print(f"Train shape: {train_gdf.shape} | Test shape: {test_gdf.shape}")
print(f"CRS: {train_gdf.crs}")
train_gdf.head(3)

### 3. Geometry Validity Check & Self-Intersection Detection
Check if any farm polygon contains topological anomalies such as self-intersections or bowtie rings.

In [ ]:
train_valid = check_geometry_validity(train_gdf)
test_valid = check_geometry_validity(test_gdf)

print(f"Train: {train_valid.sum():,} valid / {(~train_valid).sum()} invalid")
print(f"Test : {test_valid.sum():,} valid / {(~test_valid).sum()} invalid")

if (~train_valid).sum() > 0:
    invalid_indices = train_valid[~train_valid].index.tolist()
    print(f"Invalid Farm IDs: {train_gdf.loc[invalid_indices, 'FarmID'].tolist()}")

### 4. Automated Geometry Repair & Feature Enrichment
We use `shapely.make_valid` to fix topological anomalies and compute area, perimeter, compactness, centroids, and bounding boxes.

In [ ]:
train_enriched = add_geometry_features(train_gdf, repair=True)
test_enriched = add_geometry_features(test_gdf, repair=True)

print(f"Post-repair valid in Train: {train_enriched['is_valid_geom'].sum():,} / {len(train_enriched):,}")
print(f"Post-repair valid in Test : {test_enriched['is_valid_geom'].sum():,} / {len(test_enriched):,}")
train_enriched[["FarmID", "Crop", "category", "area_ha", "perimeter_m", "compactness", "centroid_lon", "centroid_lat"]].head()

### 5. Geometry Statistics Summary
Summary of farm sizes, perimeters, compactness, and vertex counts across Telangana.

In [ ]:
train_stats = geometry_stats(train_enriched)
stats_df = pd.DataFrame([train_stats]).T
stats_df.columns = ["Value"]
stats_df

### 6. Farm Area & Compactness Distributions by Crop
Compare farm boundaries, areas, and shapes across different crop categories.

In [ ]:
crop_summary = train_enriched.groupby("Crop").agg(
    farms=("FarmID", "count"),
    area_ha_mean=("area_ha", "mean"),
    area_ha_median=("area_ha", "median"),
    perimeter_m_mean=("perimeter_m", "mean"),
    compactness_mean=("compactness", "mean"),
).sort_values("farms", ascending=False)

crop_summary

### 7. Diagnostic Plots
Multi-panel visualization of farm geometry distributions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Area Distribution
sns.histplot(train_enriched["area_ha"], bins=35, kde=True, color="#2980b9", ax=axes[0])
axes[0].set_title("Farm Area Distribution (ha)", fontweight="bold")
axes[0].set_xlabel("Area (ha)")

# 2. Compactness by Crop
sns.boxplot(data=train_enriched, x="Crop", y="compactness", palette="Set2", ax=axes[1])
axes[1].set_title("Polygon Compactness by Crop", fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)

# 3. Perimeter vs Area
sample = train_enriched.sample(1000, random_state=42)
sns.scatterplot(data=sample, x="area_m2", y="perimeter_m", hue="Crop", alpha=0.6, ax=axes[2])
axes[2].set_title("Perimeter vs Area (1,000 Sample)", fontweight="bold")
axes[2].set_xlabel("Area (m²)")
axes[2].set_ylabel("Perimeter (m)")
axes[2].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

### 8. Interactive Folium Map Generation
Generate an interactive map with satellite basemap and farm polygon overlays.

In [ ]:
m = create_interactive_farm_map(
    gdf=train_enriched,
    output_html_path="../outputs/figures/telangana_farms_interactive_map.html",
    sample_size=500,
)
print("Interactive map created! Open ../outputs/figures/telangana_farms_interactive_map.html in browser.")